# r/dubai relevance extraction with DeBERTa

Cleaned from the supplied `dubai_deberta_extraction (3).ipynb`. This notebook scores submissions, keeps posts with **score > 0.4**, and retains comments linked to those posts. Comments are **not** independently classified.

Private inputs and model weights are required. No saved Reddit text or model outputs are distributed. This notebook has not been rerun on the full corpus during repository cleanup. It is not the entire extended-corpus construction pipeline; see `docs/extraction.md`.

## 1. Environment
Use a separate environment for extraction. Install `environments/extraction.txt` before running locally. In Colab, enable a GPU if available and uncomment the installation command below. The repository setup also works when opened from `notebooks/`.

In [ ]:
from pathlib import Path
import sys, subprocess

ROOT = Path.cwd()
if not (ROOT / "scripts/reproduce.py").exists() and (ROOT.parent / "scripts/reproduce.py").exists():
    ROOT = ROOT.parent
if not (ROOT / "scripts/reproduce.py").exists() and "google.colab" in sys.modules:
    ROOT = Path("/content/dld_reddit")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "https://github.com/aaronab810/dld_reddit.git", str(ROOT)], check=True)
if not (ROOT / "scripts/reproduce.py").exists():
    raise FileNotFoundError("Open this notebook from a clone of aaronab810/dld_reddit")
sys.path.insert(0, str(ROOT / "src"))

# Optional installation (uncomment if needed):
# subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(ROOT / "environments/extraction.txt")], check=True)


## 2. Configure authorized input files
Keep raw inputs and generated records under `private/`, which Git ignores. Change these paths if using a mounted Drive folder. The optional gold CSV must be the original development/threshold-selection sample, **not** an independent held-out evaluation set.

In [ ]:
import json, hashlib, importlib.metadata
import pandas as pd
from dld_reddit.extraction import validate_inputs, score_texts, select_records, LABELS, HYPOTHESIS

POSTS_PATH = ROOT / "private/dubai_posts.csv"
COMMENTS_PATH = ROOT / "private/dubai_comments.csv"
GOLD_PATH = None  # Optional: Path to the DEVELOPMENT label CSV
OUTPUT_DIR = ROOT / "private/extraction_outputs"
MODEL_ID = "MoritzLaurer/deberta-v3-large-zeroshot-v2.0"
MODEL_REVISION = "main"  # Historical revision unknown; replace with a recorded SHA to repeat a new run
THRESHOLD = 0.4


In [ ]:
posts = pd.read_csv(POSTS_PATH, dtype={"post_id": "string"}, low_memory=False)
comments = pd.read_csv(COMMENTS_PATH, dtype={"post_id": "string", "comment_id": "string"}, low_memory=False)
validate_inputs(posts, comments)
gold = None
if GOLD_PATH is not None:
    gold = pd.read_csv(GOLD_PATH)
    if not {"post_text", "actual_label"}.issubset(gold.columns):
        raise ValueError("Development labels require post_text and actual_label")
    if gold.empty or not gold.actual_label.isin([0, 1]).all():
        raise ValueError("Development labels must be complete binary 0/1 values")
    if gold.post_text.fillna("").astype(str).str.strip().eq("").any():
        raise ValueError("Blank development post_text; resolve before inference")
print(f"Input counts: {len(posts):,} posts; {len(comments):,} comments")


## 3. Load the model once
The original notebook did not record its weight revision. Resolve the configured revision now, use that exact SHA, and retain it in the local run manifest. This makes the new run traceable; it does not prove equivalence to historical model outputs.

In [ ]:
import torch
from huggingface_hub import HfApi
from transformers import pipeline

resolved_revision = HfApi().model_info(MODEL_ID, revision=MODEL_REVISION).sha
device = 0 if torch.cuda.is_available() else -1
classifier = pipeline("zero-shot-classification", model=MODEL_ID,
                      revision=resolved_revision, device=device)
print(f"Model revision: {resolved_revision}; device: {'CUDA' if device == 0 else 'CPU'}")


## 4. Optional development diagnostics
The source notebook's threshold sweep used `>=`; production selection used `>`. Preserve that distinction. These development metrics do not constitute independent validation and must not be used to tune on held-out labels.

In [ ]:
development_metrics = None
if gold is not None:
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
    gold["deberta_score"] = score_texts(gold.post_text, classifier)
    rows = []
    for threshold in [0.3, 0.4, 0.5, 0.6, 0.7]:
        predicted = gold.deberta_score.ge(threshold).astype(int)
        rows.append({"threshold": threshold, "operator": ">=", "sample": "development",
                     "accuracy": accuracy_score(gold.actual_label, predicted),
                     "precision": precision_score(gold.actual_label, predicted, zero_division=0),
                     "recall": recall_score(gold.actual_label, predicted, zero_division=0),
                     "f1": f1_score(gold.actual_label, predicted, zero_division=0)})
    development_metrics = pd.DataFrame(rows)
    display(development_metrics)
else:
    print("Development-label diagnostics skipped; no label file configured.")


## 5. Score posts and select linked comments
Scoring is sequential, uses the first 1,024 characters, the two original labels, the explicit default hypothesis template and `multi_label=False`. Keep the production threshold frozen at strict `> 0.4`. No raw rows are displayed.

In [ ]:
posts["deberta_score"] = score_texts(posts.post_text, classifier)
posts["is_real_estate"] = posts.deberta_score.gt(THRESHOLD).astype(int)
posts["prediction"] = posts.is_real_estate
selected_posts, selected_comments, joined = select_records(posts, comments, THRESHOLD)
print(f"Retained: {len(selected_posts):,} posts and {len(selected_comments):,} linked comments")
print(f"Scores exactly at threshold: {int(posts.deberta_score.eq(THRESHOLD).sum())}")


## 6. Save private outputs and provenance
These CSVs contain source records and must remain private. The manifest records the settings and software for this run. Subsequent eligibility processing and final corpus assembly are separate steps.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
posts.to_csv(OUTPUT_DIR / "dubai_posts_deberta.csv", index=False)
selected_posts.to_csv(OUTPUT_DIR / "dubai_posts_realestate_only.csv", index=False)
selected_comments.to_csv(OUTPUT_DIR / "dubai_comments_realestate_only.csv", index=False)
joined.to_csv(OUTPUT_DIR / "dubai_postscomments_realestate.csv", index=False)
if development_metrics is not None:
    development_metrics.to_csv(OUTPUT_DIR / "development_threshold_metrics.csv", index=False)

def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

manifest = {
    "model": MODEL_ID, "resolved_revision": resolved_revision,
    "historical_weight_revision_known": False,
    "labels": LABELS, "hypothesis_template": HYPOTHESIS,
    "multi_label": False, "max_characters": 1024,
    "production_threshold": THRESHOLD, "production_operator": ">",
    "development_sweep_operator": ">=", "comments_independently_classified": False,
    "inputs_sha256": {"posts": sha256(POSTS_PATH), "comments": sha256(COMMENTS_PATH)},
    "rows": {"posts": len(posts), "selected_posts": len(selected_posts), "selected_comments": len(selected_comments)},
    "versions": {name: importlib.metadata.version(name) for name in ["transformers", "torch", "pandas", "huggingface-hub", "sentencepiece"]},
}
if GOLD_PATH is not None:
    manifest["inputs_sha256"]["development_labels"] = sha256(GOLD_PATH)
(OUTPUT_DIR / "run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Private extraction outputs and run manifest saved.")
